In [ ]:
image_path = "../../img/IMG_2708.jpg"

In [ ]:
from google.cloud import vision
import io
import os
from google.protobuf.json_format import MessageToDict
from collections import defaultdict

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.expanduser("~/.secrets/vision-key.json")

class OcrClient:
    def __init__(self):
        self.client = vision.ImageAnnotatorClient()

    def extract_text_json(self, image_path):
        with io.open(image_path, 'rb') as image_file:
            content = image_file.read()

        image = vision.Image(content=content)

        # Perform OCR
        response = self.client.document_text_detection(image=image)

        if response.error.message:
            raise Exception(f"Google OCR API error: {response.error.message}")

        return response

    def parse_google_ocr_response(self, response):
        response_dict = MessageToDict(response._pb)

        lines = []
        id_counter = 1

        try:
            pages = response_dict["fullTextAnnotation"]["pages"]
        except KeyError:
            return []

        for page_idx, page in enumerate(pages):
            for block_idx, block in enumerate(page.get("blocks", [])):
                for para_idx, para in enumerate(block.get("paragraphs", [])):
                    for word_idx, word in enumerate(para.get("words", [])):
                        word_text = ''.join([s["text"] for s in word["symbols"]])
                        bbox = word["boundingBox"]["vertices"]
                        id_str = f"{page_idx + 1}.{block_idx + 1}.{id_counter}"
                        id_counter += 1

                        lines.append({
                            "id": id_str,
                            "text": word_text,
                            "bounding_box": bbox,
                            "confidence": word.get("confidence", 1.0)
                        })
        return lines

    def merge_to_lines(self, words, max_gap=30):
        lines = defaultdict(list)

        for w in words:
            y = round(w["bounding_box"][0]["y"] / max_gap)  # naive line grouping
            lines[y].append(w)

        result = []
        for i, (line_key, word_list) in enumerate(sorted(lines.items())):
            sorted_words = sorted(word_list, key=lambda w: w["bounding_box"][0]["x"])
            text = " ".join(w["text"] for w in sorted_words)
            box = [pt for w in sorted_words for pt in w["bounding_box"]]
            result.append({
                "id": f"1.1.{i+1}",
                "text": text,
                "bounding_box": box
            })
        return result



In [ ]:
import json

ocr = OcrClient()
result = ocr.extract_text_json(image_path)
lines = ocr.parse_google_ocr_response(result)
merged_lines = ocr.merge_to_lines(lines)
print(json.dumps(merged_lines, indent=2))
